# Imports

In [1]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import contractions
import tqdm
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize the tools
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

import warnings
warnings.filterwarnings("ignore")

f:\AI ENGINEER LEVEL MAP\HIGH LEVEL\Pandas for AI Engineer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4921.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Data Imports

In [2]:
df = pd.read_csv('winemag-data-130k-v2.csv', index_col=0)

In [3]:
df.head()

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


# Data Processing

In [4]:
df.drop(columns=['region_2', 'taster_name', 'taster_twitter_handle'],inplace=True)

In [5]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   country      129908 non-null  str    
 1   description  129971 non-null  str    
 2   designation  92506 non-null   str    
 3   points       129971 non-null  int64  
 4   price        120975 non-null  float64
 5   province     129908 non-null  str    
 6   region_1     108724 non-null  str    
 7   title        129971 non-null  str    
 8   variety      129970 non-null  str    
 9   winery       129971 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 54.2 MB


In [6]:
# 1. Adding default values so the LLM doesn't get confused with null value
df['designation'] = df['designation'].fillna('Standard')
df['country'] = df['country'].fillna('Unknown')
df['province'] = df['province'].fillna('Unknown')
df['region_1'] = df['region_1'].fillna('General Region')
df['variety'] = df['variety'].fillna('Others')

# 2. Calculate the median price for each specific designation
designation_medians = df.groupby('designation')['price'].transform('median')

# 3. Calculate a global median for rows where designation is also missing
global_median = df['price'].median()

df['price'] = df['price'].fillna(designation_medians).fillna(global_median)

In [7]:
to_vector = df[['title', 'description', 'variety', 'designation', 'country', 'province']]
meta_data = df[['country', 'province', 'region_1', 'variety', 'winery', 'designation', 'points', 'price']]

In [8]:
to_vector.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   title        129971 non-null  str  
 1   description  129971 non-null  str  
 2   variety      129971 non-null  str  
 3   designation  129971 non-null  str  
 4   country      129971 non-null  str  
 5   province     129971 non-null  str  
dtypes: str(6)
memory usage: 47.5 MB


In [9]:
meta_data.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   country      129971 non-null  str    
 1   province     129971 non-null  str    
 2   region_1     129971 non-null  str    
 3   variety      129971 non-null  str    
 4   winery       129971 non-null  str    
 5   designation  129971 non-null  str    
 6   points       129971 non-null  int64  
 7   price        129971 non-null  float64
dtypes: float64(1), int64(1), str(6)
memory usage: 16.1 MB


### Data Cleaned and All Set for Next Step

# Text Processing

In [10]:
def text_process(text:str):
    if not isinstance(text, str):
        return ""

    # Expand contractions
    text = contractions.fix(text)

    # Keep alphanumeric characters and basic punctuation (.,!?)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    
    return text.strip()

In [11]:
# Combine title and description for full context
# Much faster vectorized approach
to_vector['combined_text'] = (
    "Wine: " + to_vector['title'].astype(str) + 
    " | Designation: " + to_vector['designation'].astype(str) + 
    " | Variety: " + to_vector['variety'].astype(str) + 
    " | Origin: " + to_vector['country'].astype(str) + ", " + to_vector['province'].astype(str) + 
    " | Review: " + to_vector['description'].astype(str)
)

# Apply the transformation
to_vector['processed_string'] = to_vector['combined_text'].apply(text_process)

to_vector.processed_string.head()

0    Wine Nicosia 2013 Vulk Bianco  Etna  Designati...
1    Wine Quinta dos Avidagos 2011 Avidagos Red Dou...
2    Wine Rainstorm 2013 Pinot Gris Willamette Vall...
3    Wine St. Julian 2013 Reserve Late Harvest Ries...
4    Wine Sweet Cheeks 2012 Vintners Reserve Wild C...
Name: processed_string, dtype: str

# Generating Embeddings

In [12]:
# Generate the embeddings
# This creates a large numpy array where each row is a 384-dimensional vector
# Using GPU for faster computation
embeddings = model.encode(
    to_vector['processed_string'].tolist(), 
    batch_size=128, 
    show_progress_bar=True,
    convert_to_numpy=True # Keeps the output as a numpy array
)

# Add the embeddings to your dataframe
# We convert the numpy array to a list so it fits into a single column
to_vector['embeddings'] = list(embeddings)

print(f"Embedding shape: {embeddings.shape}") # Should be (number_of_rows, 384)

Batches: 100%|██████████| 1016/1016 [01:43<00:00,  9.80it/s]


Embedding shape: (129971, 384)


# Saving the data

In [13]:
# Save the array to a .npy file
np.save("wine_review_embeddings.npy", embeddings)

print(f"Vectors saved! Shape: {embeddings.shape}")

Vectors saved! Shape: (129971, 384)


In [15]:
# Save as a standard CSV
meta_data.to_parquet("wine_review_metadata.parquet", index=False)

print("Metadata (Text/IDs) saved to wine_review_metadata.parquet")


Metadata (Text/IDs) saved to wine_review_metadata.parquet
